# **BƯỚC 1: THIẾT LẬP HẠ TẦNG MÔ PHỎNG ns-3 & MIDDLEWARE TRÊN GOOGLE COLAB**

* Tại sao không dùng ns-3? ns-3 quá nặng nề và phức tạp để thiết lập các thông số như BSS Color hay OBSS_PD của Wi-Fi 6. Chúng ta sẽ dùng Gymnasium để tự code một môi trường mô phỏng toán học (Digital Twin) dựa trên phương trình Shannon và mô hình suy hao.
* Mục tiêu: Xây dựng một gym.Env ảo trên Colab để pre-train mô hình PPO/SAC trước khi nạp vào phần cứng thật.
* Cách thực hiện & Cấu trúc Code:
    1. Cài đặt môi trường cơ bản trên Colab:
    `!pip install gymnasium stable-baselines3 paramiko scapy`.
    2. Xây dựng lớp `DigitalTwinEnv(gym.Env)`:
        * Tích hợp toán học vật lý vào hàm `step()`: Tính SNR dựa trên công suất phát ($P_{tx}$) và suy hao khoảng cách. Tính thông lượng (Throughput) dự kiến theo định lý Shannon mở rộng.   
        * Mô phỏng MAC-Sensing: Thêm yếu tố ngẫu nhiên để mô phỏng Tỷ lệ rớt gói (Packet Loss) và Xác suất va chạm ($p_c$) tăng lên khi mở rộng băng thông (từ 20MHz lên 80MHz).   

In [1]:
print("=== THIẾT LẬP HẠ TẦNG DIGITAL TWIN (BẢN SAO SỐ) VÀ RL ===")

# 1. Cài đặt các thư viện AI và Môi trường
print("\n[*] Đang cài đặt Gymnasium, Stable-Baselines3 và các thư viện cần thiết...")
!pip install -q gymnasium stable-baselines3[extra] numpy pandas matplotlib paramiko scapy

import gymnasium as gym
from gymnasium import spaces
import numpy as np
import math

print("\n[+] Đã cài đặt xong! Khởi tạo môi trường Digital Twin...")

# 2. Xây dựng Môi trường Bản sao số (DigitalTwinEnv)
class DigitalTwinEnv(gym.Env):
    """
    Môi trường giả lập mạng Wi-Fi dựa trên MAC-Layer Proxy Metrics và toán học vật lý.
    Dành cho việc huấn luyện Offline trước khi đưa xuống Router OpenWrt thật.
    """
    def __init__(self):
        super(DigitalTwinEnv, self).__init__()

        # --- A. KHÔNG GIAN HÀNH ĐỘNG (ACTION SPACE) ĐA HÌNH ---
        # Dựa trên chiến thuật Tối ưu hạ tầng vô tuyến
        self.action_space = spaces.Dict({
            'channel': spaces.Discrete(4),          # Kênh (VD: 36, 40, 44, 48)
            'bandwidth': spaces.Discrete(3),        # Băng thông: 0=20MHz, 1=40MHz, 2=80MHz
            'tx_power': spaces.Box(low=8.0, high=14.0, shape=(1,), dtype=np.float32), # Cân bằng công suất 8-14dBm
            'mcs_ceiling': spaces.Discrete(12),     # Khóa trần MCS (0-11)
            'beacon_int': spaces.Discrete(2),       # Interval: 0=80ms (Ra chơi), 1=300ms (Trong tiết)
            'obss_pd_level': spaces.Box(low=-82.0, high=-62.0, shape=(1,), dtype=np.float32) # Ngưỡng Spatial Reuse
        })

        # --- B. KHÔNG GIAN TRẠNG THÁI (STATE SPACE) THEO MAC-SENSING ---
        # [RSSI, p_c (Va chạm), Retry_Rate, Airtime_Util, STA_Count, Hour, Minute, Is_Break_Time]
        self.observation_space = spaces.Box(
            low=np.array([-90.0, 0.0, 0.0, 0.0, 0, 0, 0, 0]),
            high=np.array([-30.0, 1.0, 1.0, 1.0, 100, 23, 59, 1]),
            dtype=np.float32
        )

        self.state = None
        self.current_step = 0
        self.max_steps = 100 # Số bước tối đa cho 1 episode

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = 0
        # Giả lập trạng thái khởi đầu ngẫu nhiên
        self.state = np.array([-70.0, 0.1, 0.05, 0.4, 20, 8, 15, 0], dtype=np.float32)
        return self.state, {}

    def step(self, action):
        self.current_step += 1

        # --- C. TOÁN HỌC VẬT LÝ VÀ MÔ PHỎNG (SẼ CODE CHI TIẾT SAU) ---
        # (Ở đây ta tạm dùng logic giả lập để mô hình có thể chạy được)

        # Phân tích Action
        bw_action = action['bandwidth']
        tx_power_action = action['tx_power'][0]
        is_break_time = self.state[7]

        # Giả lập phản ứng môi trường
        throughput = 50.0 # Tốc độ cơ sở (Mbps)
        latency = 20.0    # Trễ cơ sở (ms)
        packet_loss = 0.0

        # Logic 1: Bandwidth Fallback & Nhiễu
        if bw_action == 2: # Nếu AI tham lam chọn 80MHz
            throughput += 30
            latency += 50 # Bị nhiễu CCI cao, ping vọt lên
            packet_loss = 0.08
        elif bw_action == 0: # An toàn ở 20MHz
            latency -= 10
            packet_loss = 0.01

        # Logic 2: Time-Aware Context (Tối ưu theo thời khóa biểu)
        if is_break_time == 0 and action['beacon_int'] == 1:
            throughput += 5 # Thưởng vì giảm overhead trong giờ học
        elif is_break_time == 1 and action['beacon_int'] == 0:
            latency -= 5 # Thưởng vì giúp Roaming mượt giờ ra chơi

        # --- D. TÍNH TOÁN REWARD ---
        # R_t = alpha*Throughput - beta*Latency - Penalty(Loss)
        reward = (0.4 * throughput) - (0.3 * latency)

        # Phạt nặng nếu rớt gói > 5%
        if packet_loss > 0.05:
            reward -= 50.0

        # Cập nhật State mới (Giả lập)
        self.state = np.array([-65.0, packet_loss, packet_loss, 0.6, 25, 8, 20, 0], dtype=np.float32)

        terminated = bool(self.current_step >= self.max_steps)
        truncated = False

        return self.state, reward, terminated, truncated, {"throughput": throughput, "latency": latency}

# Khởi tạo thử môi trường
env = DigitalTwinEnv()
obs, info = env.reset()
print("\n✅ Khởi tạo Môi trường Bản sao số thành công!")
print("Trạng thái ban đầu (State):", obs)
print("Không gian Hành động (Action Space):", env.action_space)

=== THIẾT LẬP HẠ TẦNG DIGITAL TWIN (BẢN SAO SỐ) VÀ RL ===

[*] Đang cài đặt Gymnasium, Stable-Baselines3 và các thư viện cần thiết...
error: incomplete-download

× Download failed because not enough bytes were received (158.3 MB/554.6 MB)
╰─> URL: https://files.pythonhosted.org/packages/fd/df/23c69e9b9fd19fe6563422f1bc59f89601bda47234a89a1cf432cfde3aee/torch-2.14.1-cp314-cp314-manylinux_2_28_x86_64.whl

note: This is an issue with network connectivity, not pip.
hint: Consider using --resume-retries to enable download resumption.


ModuleNotFoundError: No module named 'gymnasium'

# **BƯỚC 2: Định hình Không gian Đa hình (Polymorphic Space) & Ràng buộc**
 * Mục tiêu: Khai báo Không gian Trạng thái (State) và Hành động (Action) phản ánh đúng các chiến thuật tầng PHY/MAC mà chúng ta đã chốt (Bandwidth Elasticity, MCS Ceiling, v.v.).
 * Cách thực hiện & Cấu trúc Code:
    1. State (Trạng thái - $\mathcal{S}$): Code mảng đầu vào nhận các Proxy Metrics: `[RSSI, p_c, Retry_Rate, Airtime_Util, STA_Count, V_time]`. Trong đó `V_time = [Hour, Minute, Is_Break_Time]` để AI nhận thức thời khóa biểu.
    2. Action (Hành động đa chiều - $\mathcal{A}$): Sử dụng `spaces.Dict` để bao quát toàn bộ vũ khí của AI:
        * `Channel`: Chọn kênh (Ví dụ: 36, 40, 44).
        * Bandwidth: Co giãn 20/40/80 MHz.
        * `Tx_Power`: Khớp công suất 8 - 14 dBm (Tx Power Matching).
        * `MCS_Ceil`: Khóa trần tốc độ để chống Fading.
    3. Reward (Phần thưởng - $\mathcal{R}$): Code hàm Reward đa mục tiêu:
    * Thưởng ($\alpha \times Throughput$) và Phạt ($\beta \times Latency$).
    * Phạt cực nặng (Penalty): Trừ điểm khủng khiếp nếu Tỷ lệ rớt gói $> 5\%$ hoặc AI chọn Action gây đứt mạng.

In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np

print("=== BƯỚC 2: KHỞI TẠO LỚP BỌC ĐA HÌNH (POLYMORPHIC WRAPPER) ===")

# --- 1. LỚP MÔI TRƯỜNG LÕI (CORE ENVIRONMENT) ---
class CoreDigitalTwinEnv(gym.Env):
    """Môi trường gốc, chứa toàn bộ vũ khí tối đa của hệ thống"""
    def __init__(self):
        super().__init__()

        # Không gian hành động Max-level (Bao gồm cả Wi-Fi 6)
        self.action_space = spaces.Dict({
            'channel': spaces.Discrete(3),          # [1, 6, 11] cho 2.4GHz hoặc [36, 40, 44] cho 5GHz
            'bandwidth': spaces.Discrete(3),        # [20MHz, 40MHz, 80MHz]
            'tx_power': spaces.Box(low=8.0, high=14.0, shape=(1,), dtype=np.float32),
            'mcs_ceiling': spaces.Discrete(12),     # MCS 0-11
            'beacon_int': spaces.Discrete(2),       # [80ms, 300ms]
            'obss_pd_level': spaces.Box(low=-82.0, high=-62.0, shape=(1,), dtype=np.float32), # Chỉ có ở Wi-Fi 6
            'vht_beamform': spaces.Discrete(2)      # 0: Tắt, 1: Bật (Chỉ Wi-Fi 5/6)
        })

        # Không gian Trạng thái (MAC-Sensing)
        # [RSSI, p_c (Va chạm), Retry_Rate, Airtime_Util, STA_Count, Hour, Minute, Is_Break_Time]
        self.observation_space = spaces.Box(
            low=np.array([-90.0, 0.0, 0.0, 0.0, 0, 0, 0, 0], dtype=np.float32),
            high=np.array([-30.0, 1.0, 1.0, 1.0, 100, 23, 59, 1], dtype=np.float32)
        )
        self.current_step = 0
        self.max_steps = 100

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = 0
        return np.array([-70.0, 0.1, 0.05, 0.4, 20, 8, 15, 0], dtype=np.float32), {}

    def step(self, action):
        self.current_step += 1

        # (Logic giả lập tạm thời để pass qua Bước 2)
        reward = np.random.uniform(10, 50)
        state = self.observation_space.sample()
        terminated = bool(self.current_step >= self.max_steps)

        return state, reward, terminated, False, {}

# --- 2. LỚP BỌC ĐA HÌNH (POLYMORPHIC WRAPPER) ---
class HardwareAwareWrapper(gym.ActionWrapper):
    """
    Lớp bọc thông minh: Tự động khóa các Action không được hỗ trợ bởi Router.
    """
    def __init__(self, env, hardware_type="WIFI_6"):
        super().__init__(env)
        self.hardware_type = hardware_type

        # Nếu là Wi-Fi 4 (Router cũ), cắt bỏ các tính năng cao cấp
        if self.hardware_type == "WIFI_4_LEGACY":
            print("[!] CẢNH BÁO: Đã phát hiện Router Wi-Fi 4. Kích hoạt chế độ Fallback.")
            # Tạo Action Space mới, loại bỏ OBSS_PD, Beamform, MCS>7, Băng thông 80MHz
            self.action_space = spaces.Dict({
                'channel': spaces.Discrete(3),
                'bandwidth': spaces.Discrete(2), # Chỉ 20/40MHz
                'tx_power': spaces.Box(low=8.0, high=14.0, shape=(1,), dtype=np.float32),
                'mcs_ceiling': spaces.Discrete(8), # Chỉ MCS 0-7 (Chuẩn 802.11n)
                'beacon_int': spaces.Discrete(2)
            })
        else:
            print("[+] Đã phát hiện Router Wi-Fi 6 (MediaTek). Kích hoạt toàn bộ tính năng RRM.")

    def action(self, action):
        """
        Hàm này chặn Action từ AI (Stable-Baselines3) gửi xuống Env.
        Nếu phần cứng là Wi-Fi 4, nó sẽ tự động chèn các giá trị mặc định cho các tính năng bị thiếu
        để môi trường lõi (Core Env) không bị lỗi.
        """
        if self.hardware_type == "WIFI_4_LEGACY":
            # Chèn các giá trị cứng (Hardcoded) cho các tính năng không được hỗ trợ
            processed_action = action.copy()
            processed_action['obss_pd_level'] = np.array([-82.0], dtype=np.float32) # Mặc định
            processed_action['vht_beamform'] = 0 # Tắt Beamforming
            # Đảm bảo băng thông không vượt quá 40MHz
            if processed_action['bandwidth'] > 1:
                processed_action['bandwidth'] = 1
            return processed_action

        return action # Nếu là Wi-Fi 6, giữ nguyên lệnh của AI

# --- 3. KIỂM THỬ KHẢ NĂNG CO GIÃN CỦA WRAPPER ---

print("\n--- TEST KỊCH BẢN 1: MÔI TRƯỜNG WI-FI 6 (Xiaomi AX3000T) ---")
env_wifi6 = HardwareAwareWrapper(CoreDigitalTwinEnv(), hardware_type="WIFI_6")
print("Action Space (Wi-Fi 6):", list(env_wifi6.action_space.keys()))

print("\n--- TEST KỊCH BẢN 2: MÔI TRƯỜNG WI-FI 4 (Router Cũ mượn đi thi) ---")
env_wifi4 = HardwareAwareWrapper(CoreDigitalTwinEnv(), hardware_type="WIFI_4_LEGACY")
print("Action Space (Wi-Fi 4):", list(env_wifi4.action_space.keys()))

print("\n✅ BƯỚC 2 HOÀN TẤT. Hệ thống đã có khả năng tự nhận thức phần cứng (Hardware-Aware)!")

# **BƯỚC 3: Huấn luyện (Offline Training) & Khắc phục "Ping-pong"**

* Mục tiêu: Dạy AI trên Bản sao số (Digital Twin) trên GPU của Google Colab cho đến khi đường cong hội tụ (Reward Curve) ổn định.
* Cách thực hiện & Cấu trúc Code:
  1. Khởi tạo tác tử PPO hoặc SAC từ thư viện `stable-baselines3`.
  2. Chạy vòng lặp huấn luyện khoảng 500,000 timesteps.
  3. Chiến thuật Hysteresis (Chống Ping-pong): Trong hàm tính Reward, thêm một khoản trừ điểm (penalty) nhỏ mỗi khi AI quyết định thay đổi trạng thái (ví dụ: đang kênh 36 đổi sang kênh 40). Điều này ép AI chỉ đổi kênh/băng thông khi thực sự mang lại lợi ích lớn, tránh việc cấu hình liên tục làm đứt mạng học sinh (wifi reload liên tục).
  4. Giám sát bằng TensorBoard: Xuất biểu đồ Loss và Reward để đưa vào báo cáo IMRAD.

In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.monitor import Monitor
from typing import Callable
import os

print("=== BƯỚC 3: HUẤN LUYỆN ĐẶC NHIỆM (REWARD SCALING & NETWORK TUNING) ===")

# --- 1. LỊCH TRÌNH HỌC TẬP TUYẾN TÍNH (LINEAR LR SCHEDULE) ---
def linear_schedule(initial_value: float) -> Callable[[float], float]:
    """
    Hàm giảm tốc độ học (Learning Rate) dần dần về 0 theo tiến độ training.
    Giúp AI hội tụ ổn định ở những steps cuối, tránh dao động (overfitting).
    """
    def func(progress_remaining: float) -> float:
        # progress_remaining đi từ 1.0 (bắt đầu) xuống 0.0 (kết thúc)
        return progress_remaining * initial_value
    return func

class UltimateDigitalTwinEnv(gym.Env):
    def __init__(self, hardware_type="WIFI_6"):
        super().__init__()
        self.hardware_type = hardware_type
        self.action_space = spaces.MultiDiscrete([3, 3, 7, 12, 2, 5, 2])
        self.observation_space = spaces.Box(
            low=np.array([-90.0, 0.0, 0.0, 0.0, 0, 0, 0, 0], dtype=np.float32),
            high=np.array([-30.0, 1.0, 1.0, 1.0, 300, 23, 59, 1], dtype=np.float32)
        )
        self.current_step = 0
        self.max_steps = 300
        self.previous_action = None
        self.disaster_timer = 0
        self.disaster_type = 0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = 0
        self.previous_action = None
        self.disaster_timer = 0
        self.disaster_type = 0
        return np.array([-70.0, 0.1, 0.05, 0.4, 20, 8, 15, 0], dtype=np.float32), {}

    def step(self, action):
        self.current_step += 1
        channel_idx = action[0]
        bw_idx = action[1]

        # DOMAIN RANDOMIZATION (Giữ nguyên cơ chế nhưng làm mượt logic)
        if self.disaster_timer <= 0:
            if np.random.rand() < 0.15:
                self.disaster_timer = np.random.randint(20, 50)
                self.disaster_type = np.random.choice([1, 2])
            else:
                self.disaster_type = 0
        else:
            self.disaster_timer -= 1

        noise_level = 0.1 if channel_idx == 0 else (0.2 if channel_idx == 1 else 0.4)
        actual_sta = np.random.randint(15, 35)

        if self.disaster_type == 1 and channel_idx == 0:
            noise_level = 0.95
        elif self.disaster_type == 2:
            actual_sta = np.random.randint(120, 180)

        base_throughput = 50.0 if bw_idx == 0 else (100.0 if bw_idx == 1 else 200.0)
        p_c = noise_level + (actual_sta * 0.005)

        if bw_idx == 2 and actual_sta > 100:
            p_c += 0.5
        elif bw_idx == 1 and actual_sta > 100:
            p_c += 0.2

        p_c = min(1.0, p_c)
        actual_throughput = base_throughput * (1.0 - p_c)

        # --- CẢI TIẾN: REWARD SCALING CHUẨN MỰC ---
        # Đưa phần thưởng về dải quy mô nhỏ để Gradient không bị "nổ"
        # Throughput cao nhất ~200 -> Scale xuống ~2.0
        # Va chạm p_c cao nhất 1.0 -> Phạt tối đa -5.0
        norm_throughput = actual_throughput / 100.0
        penalty_collision = 5.0 * p_c

        base_reward = norm_throughput - penalty_collision

        action_penalty = 0.0
        if self.previous_action is not None:
            if channel_idx != self.previous_action[0]:
                action_penalty -= 0.5 # Scale lại mức phạt Hysteresis (tương đương 5đ cũ)
            if bw_idx != self.previous_action[1]:
                action_penalty -= 0.3

        self.previous_action = action.copy()
        final_reward = base_reward + action_penalty

        state = np.array([-65.0, p_c, 0.05, 0.4, actual_sta, 8, 15, 0], dtype=np.float32)
        terminated = bool(self.current_step >= self.max_steps)

        return state, float(final_reward), terminated, False, {}

# --- HUẤN LUYỆN VỚI SIÊU THAM SỐ (HYPERPARAMETER TUNING) ---
log_dir = "./wifi_rl_logs/"
os.makedirs(log_dir, exist_ok=True)

env = UltimateDigitalTwinEnv(hardware_type="WIFI_6")
env = Monitor(env, log_dir)

print("\n[*] Đang khởi tạo mô hình PPO với Mạng Nơ-ron lớn [256, 256]...")

# 2. MỞ RỘNG MẠNG NƠ RON: Thêm não bộ để AI xử lý dữ liệu phức tạp
policy_kwargs = dict(net_arch=[256, 256])

# 3. TỐI ƯU HÓA PPO PARAMS
model = PPO("MlpPolicy", env,
            learning_rate=linear_schedule(0.0005), # Bắt đầu ở 5e-4, giảm dần về 0
            n_steps=2048,           # Thu thập nhiều dữ liệu hơn trước khi update (chống nhiễu)
            batch_size=256,         # Batch size lớn hơn giúp gradient ổn định
            ent_coef=0.005,         # Ép AI bớt tò mò, tập trung vào chiến thuật đúng
            policy_kwargs=policy_kwargs,
            verbose=0,
            tensorboard_log=log_dir,
            device='cpu')

eval_callback = EvalCallback(env, best_model_save_path='./saved_models/',
                             log_path=log_dir, eval_freq=2000,
                             deterministic=True, render=False)

print("\n[*] TIẾN HÀNH HUẤN LUYỆN (300,000 TIMESTEPS)...")
model.learn(total_timesteps=300000, callback=eval_callback)

print("\n✅ HUẤN LUYỆN HOÀN TẤT! 'BEST_MODEL.ZIP' SIÊU CẤP ĐÃ SẴN SÀNG!")

# **BƯỚC 4: Tích hợp Lớp bọc An toàn (Dual Fail-Safe Wrapper) cho Sim-to-Real**

* Mục tiêu: Đảm bảo khi mang mô hình từ Colab xuống Router thật, AI không thể "nổi điên" phá sập mạng trường học.
* Cách thực hiện & Cấu trúc Code:
  1. Trên Edge Server (Laptop), viết một Python Daemon Script chạy song song.
  2. Soft Fail-safe: Daemon liên tục ping đến Router. Nếu Ping > 500ms quá 30 giây (mạng bị nghẽn do AI chọn sai tham số), script tự động gọi kill để ngắt tiến trình AI, và đẩy lệnh SSH `wifi reload` để reset router về mặc định.
  3. Action Masking (Bộ lọc Lệnh): Viết đè hàm `predict` của AI. Trước khi lệnh SSH được gửi đi, code sẽ kiểm tra: Nếu AI đòi đặt `Tx_Power = 30dBm` (vi phạm luật EIRP), lệnh đó sẽ bị chặn lại và thay thế bằng mức an toàn (ví dụ 20dBm).

In [ ]:
import time
import subprocess
import paramiko

print("=== BƯỚC 4: KHỞI TẠO LỚP BỌC AN TOÀN (FAIL-SAFE CONTROLLER) ===")

class FailSafeController:
    """
    Module độc lập chịu trách nhiệm giám sát và bảo vệ Router thật khỏi các quyết định sai lầm của AI.
    """
    def __init__(self, router_ip="192.168.1.1", router_user="root", router_pass="your_password"):
        self.router_ip = router_ip
        self.router_user = router_user
        self.router_pass = router_pass

        # Cấu hình ngưỡng an toàn
        self.MAX_PING_THRESHOLD_MS = 500  # Ngưỡng Ping tối đa cho phép (500ms)
        self.MAX_ALLOWED_TX_POWER = 20    # Giới hạn công suất phát tối đa (dBm)
        self.ALLOWED_CHANNELS = [36, 40, 44, 48, 149, 153, 157, 161] # Các kênh 5GHz hợp lệ

        print(f"[*] FailSafe Controller đã sẵn sàng. Giám sát IP: {self.router_ip}")

    # --- LỚP BẢO VỆ 1: ACTION MASKING (LỌC LỆNH TỪ AI) ---
    def validate_and_filter_action(self, raw_action):
        """
        Kiểm tra và 'gọt giũa' lệnh của AI trước khi gửi xuống Router.
        """
        safe_action = raw_action.copy()

        # 1. Kiểm tra Công suất phát (Tx Power)
        # AI có thể xuất ra số float kỳ lạ, ta ép kiểu và giới hạn
        requested_tx_power = int(safe_action.get('tx_power', 10))
        if requested_tx_power > self.MAX_ALLOWED_TX_POWER:
            print(f"[!] CẢNH BÁO FAIL-SAFE: AI yêu cầu Tx Power {requested_tx_power}dBm vượt ngưỡng an toàn! Ép về {self.MAX_ALLOWED_TX_POWER}dBm.")
            safe_action['tx_power'] = self.MAX_ALLOWED_TX_POWER
        elif requested_tx_power < 8: # Mức tối thiểu để sóng không bị chết
            safe_action['tx_power'] = 8

        # 2. Kiểm tra Kênh (Channel)
        requested_channel = safe_action.get('channel', 36)
        if requested_channel not in self.ALLOWED_CHANNELS:
            print(f"[!] CẢNH BÁO FAIL-SAFE: Kênh {requested_channel} không hợp lệ! Ép về Kênh 36.")
            safe_action['channel'] = 36

        # Các thông số khác (MCS, Băng thông) có thể lọc tương tự...

        return safe_action

    # --- LỚP BẢO VỆ 2: WATCHDOG (GIÁM SÁT HẬU QUẢ VÀ CỨU HỘ MỀM) ---
    def measure_ping(self):
        """Đo độ trễ Ping tới Router để kiểm tra sức khỏe mạng"""
        try:
            # Gửi 1 gói tin ping, chờ tối đa 1 giây (-W 1)
            output = subprocess.check_output(
                ["ping", "-c", "1", "-W", "1", self.router_ip],
                stderr=subprocess.STDOUT, universal_newlines=True
            )
            # Dùng regex hoặc cắt chuỗi để lấy ms (Đơn giản hóa cho ví dụ)
            if "time=" in output:
                time_ms = float(output.split("time=")[1].split(" ms")[0])
                return time_ms
            return 9999.0
        except subprocess.CalledProcessError:
            # Lỗi ping (mạng đứt hoặc rớt gói hoàn toàn)
            return 9999.0

    def trigger_soft_rescue(self):
        """
        Cứu hộ Mềm: Kết nối SSH và ép Router quay về cấu hình an toàn (Auto-Channel).
        """
        print("\n[!!!] KÍCH HOẠT CỨU HỘ KHẨN CẤP (SOFT FAIL-SAFE) [!!!]")
        print("[*] Đang kết nối SSH để reset cấu hình Wi-Fi...")

        ssh = paramiko.SSHClient()
        ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
        try:
            ssh.connect(self.router_ip, username=self.router_user, password=self.router_pass, timeout=5)

            # Lệnh OpenWrt: Đổi kênh về Auto, công suất về Auto, và khởi động lại Wi-Fi
            rescue_command = (
                "uci set wireless.radio0.channel='auto' && "
                "uci set wireless.radio0.txpower='auto' && "
                "uci commit wireless && wifi reload"
            )
            ssh.exec_command(rescue_command)
            print("[+] Đã nạp cấu hình cứu hộ! Mạng sẽ khôi phục trong 5-10 giây.")
        except Exception as e:
            print(f"[-] THẤT BẠI KHI CỨU HỘ QUA SSH: {e}")
            print("[!] CẦN KÍCH HOẠT CỨU HỘ CỨNG (HARD FAIL-SAFE QUA Ổ CẮM THÔNG MINH)!")
            # (Đoạn này sẽ gọi Local API của Sonoff S26 R2 để cắt điện)
        finally:
            ssh.close()

    def watchdog_loop(self):
        """
        Vòng lặp chạy ngầm để giám sát mạng liên tục.
        (Trong thực tế sẽ chạy bằng Thread riêng)
        """
        print("[*] Watchdog đang giám sát sức khỏe mạng...")
        high_ping_count = 0

        # Giả lập chạy 5 lần đo
        for i in range(5):
            ping_ms = self.measure_ping()
            print(f"    - Lần đo {i+1}: Ping = {ping_ms:.2f} ms")

            if ping_ms > self.MAX_PING_THRESHOLD_MS:
                high_ping_count += 1
                print(f"    [!] Cảnh báo: Ping quá cao ({high_ping_count}/3 lần liên tiếp)")
            else:
                high_ping_count = 0 # Reset nếu mạng ổn định lại

            # Nếu 3 lần liên tiếp Ping > 500ms hoặc đứt mạng -> Kích hoạt cứu hộ
            if high_ping_count >= 3:
                self.trigger_soft_rescue()
                break # Dừng vòng lặp để cứu hộ

            time.sleep(1) # Đợi 1 giây giữa các lần ping

# --- TEST LỚP BỌC AN TOÀN ---
print("\n--- TEST KỊCH BẢN 1: ACTION MASKING ---")
fail_safe = FailSafeController(router_ip="127.0.0.1") # Dùng localhost để test
ai_action = {'channel': 999, 'tx_power': 30, 'bandwidth': 80} # AI bị "ngáo", xuất action vô lý
print("Action từ AI:", ai_action)
safe_action = fail_safe.validate_and_filter_action(ai_action)
print("Action đã gọt giũa an toàn:", safe_action)

print("\n--- TEST KỊCH BẢN 2: WATCHDOG GIÁM SÁT ---")
# Lưu ý: Vì ta đang test trên localhost/Colab, lệnh ping có thể luôn < 1ms hoặc thất bại
# Đoạn code này chỉ minh họa luồng hoạt động
fail_safe.watchdog_loop()

print("\n✅ BƯỚC 4 HOÀN TẤT. Hệ thống đã có 'Vệ sĩ' bảo vệ!")

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

print("=== BƯỚC 4.5: TRỰC QUAN HÓA KẾT QUẢ HUẤN LUYỆN (LEARNING CURVES) ===")

# --- 1. ĐỌC DỮ LIỆU TỪ FILE LOG ---
log_dir = "./wifi_rl_logs/"
monitor_file = os.path.join(log_dir, "monitor.csv")

if not os.path.exists(monitor_file):
    print(f"[!] Lỗi: Không tìm thấy file {monitor_file}. Bạn đã chạy Bước 3 chưa?")
else:
    print(f"[*] Đang tải dữ liệu từ {monitor_file}...")

    # Bỏ qua 1 dòng đầu tiên (chứa metadata của stable-baselines3)
    df = pd.read_csv(monitor_file, skiprows=1)

    # df chứa các cột: r (reward), l (length - số steps trong episode), t (time)
    episodes = np.arange(len(df))
    rewards = df['r'].values
    lengths = df['l'].values

    # --- 2. HÀM LÀM MƯỢT (SMOOTHING) ĐỂ BIỂU ĐỒ ĐẸP HƠN ---
    def smooth(scalars, weight=0.85):
        """
        Hàm làm mượt dữ liệu (Exponential Moving Average) giống TensorBoard.
        Giúp đường cong hội tụ mượt mà, dễ nhìn xu hướng hơn.
        """
        last = scalars[0]
        smoothed = []
        for point in scalars:
            smoothed_val = last * weight + (1 - weight) * point
            smoothed.append(smoothed_val)
            last = smoothed_val
        return smoothed

    rewards_smoothed = smooth(rewards)

    # --- 3. VẼ BIỂU ĐỒ ---
    # Thiết lập style đồ thị chuẩn báo cáo khoa học
    plt.style.use('seaborn-v0_8-darkgrid')
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 10))

    # Biểu đồ 1: Episode Reward (Đường cong hội tụ)
    ax1.plot(episodes, rewards, alpha=0.3, color='royalblue', label='Raw Reward')
    ax1.plot(episodes, rewards_smoothed, color='darkblue', linewidth=2, label='Smoothed Trend (EMA)')
    ax1.set_title("Biểu đồ 1: Sự hội tụ Phần thưởng (Reward Convergence) của Tác tử PPO", fontsize=14, fontweight='bold')
    ax1.set_xlabel("Số vòng lặp (Episodes)", fontsize=12)
    ax1.set_ylabel("Phần thưởng Tích lũy (Episode Reward)", fontsize=12)
    ax1.legend(loc="lower right")
    ax1.grid(True, linestyle='--', alpha=0.7)

    # Đánh dấu điểm Reward hội tụ
    ax1.axhline(y=16000, color='red', linestyle=':', label='Optimal Reward Target')

    # Biểu đồ 2: Episode Length
    ax2.plot(episodes, lengths, color='forestgreen', linewidth=2)
    ax2.set_title("Biểu đồ 2: Độ dài Vòng lặp (Episode Length)", fontsize=14, fontweight='bold')
    ax2.set_xlabel("Số vòng lặp (Episodes)", fontsize=12)
    ax2.set_ylabel("Số Steps", fontsize=12)
    ax2.grid(True, linestyle='--', alpha=0.7)
    # Vì môi trường của ta cố định max_steps = 200, đường này sẽ là một đường thẳng ngang.
    ax2.set_ylim(0, 250)

    # Căn chỉnh và hiển thị
    plt.tight_layout()

    # --- 4. LƯU BIỂU ĐỒ THÀNH FILE ẢNH CHẤT LƯỢNG CAO ---
    output_image = "./wifi_rl_logs/learning_curve_plot.png"
    plt.savefig(output_image, dpi=300, bbox_inches='tight')
    print(f"\n✅ Đã vẽ xong biểu đồ! Đã lưu ảnh chất lượng cao (300 DPI) tại: {output_image}")

    # Hiển thị trực tiếp trên Colab
    plt.show()

In [ ]:
import numpy as np
import pandas as pd
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import os

print("=== BƯỚC 4.6: KHKT STRESS TEST - KAGGLE TRACE-DRIVEN SIMULATION ===")

# --- 1. HÀM TRÍCH XUẤT VẾT DỮ LIỆU TỪ KAGGLE (ETL PIPELINE) ---
def prepare_kaggle_traces(wifi_path="/content/data/wifi_dataset_cleaned.csv", uji_path="/content/data/TrainingData.csv"):
    """
    Đọc dữ liệu thực tế từ Kaggle, chuẩn hóa về dải [0, 1] đối với Nhiễu
    và trích xuất số lượng thiết bị để tạo kịch bản 300 bước Thảm họa.
    """
    print(f"[*] Đang xử lý dữ liệu Kaggle...")

    # 1A. Xử lý dữ liệu UJIIndoorLoc (Mật độ người dùng - Bão Roaming)
    try:
        uji_df = pd.read_csv(uji_path)
        # Giả lập gom nhóm theo thời gian/index để tạo mật độ trạm (STA_Count)
        # Tùy thuộc data, ta mô phỏng bằng cách đếm số dòng trong các cửa sổ trượt
        sta_counts = np.histogram(uji_df.index, bins=300)[0]
        # Khai thác những lúc vắng vẻ (Bình thường) và những lúc nghẽn mạng (Bão)
        normal_sta_traces = sorted(sta_counts)[:200]
        roaming_storm_traces = sorted(sta_counts)[-100:]

        # Cân chỉnh lại cho hợp lý với bài toán phòng học (20 -> 150 thiết bị)
        normal_sta_traces = np.clip(normal_sta_traces, 10, 40)
        roaming_storm_traces = np.clip(roaming_storm_traces * 2, 80, 200)
        print("  [+] Đã trích xuất dấu vết Bão Roaming từ UJIIndoorLoc.")
    except Exception as e:
        print(f"  [!] Lỗi đọc UJIIndoorLoc: {e}. Đang dùng phân phối Beta thay thế.")
        normal_sta_traces = np.random.randint(15, 35, 200)
        roaming_storm_traces = np.random.randint(100, 180, 100)

    # 1B. Xử lý dữ liệu Wi-Fi 2.4GHz RF Telemetry (Nhiễu Kênh)
    try:
        wifi_df = pd.read_csv(wifi_path)
        scaler = MinMaxScaler()
        # Tự động tìm cột chứa dữ liệu số (như Noise, Interference, Signal)
        numeric_cols = wifi_df.select_dtypes(include=np.number).columns
        # Lấy cột số liệu đầu tiên làm cơ sở để tạo nhiễu, chuẩn hóa về [0, 1]
        noise_data = scaler.fit_transform(wifi_df[numeric_cols[-1]].values.reshape(-1, 1)).flatten()

        low_noise_traces = sorted(noise_data)[:200] # Nhiễu nền thấp
        high_noise_traces = sorted(noise_data)[-100:] # Cực đại nhiễu từ thiết bị lậu
        print("  [+] Đã trích xuất dấu vết Nhiễu từ Wi-Fi RF Telemetry.")
    except Exception as e:
        print(f"  [!] Lỗi đọc RF Telemetry: {e}. Đang dùng phân phối Gamma thay thế.")
        low_noise_traces = np.random.uniform(0.05, 0.2, 200)
        high_noise_traces = np.random.uniform(0.8, 0.99, 100)

    # 1C. Lắp ráp kịch bản Hỗn loạn (Chaos Scenarios)
    data = []
    for step in range(300):
        if step < 100:
            # PHASE 1: MẠNG ỔN ĐỊNH
            data.append({
                'step': step,
                'noise_ch0': np.random.choice(low_noise_traces),
                'noise_ch1': np.random.choice(low_noise_traces),
                'noise_ch2': np.random.choice(low_noise_traces),
                'sta_count': int(np.random.choice(normal_sta_traces))
            })
        elif step < 200:
            # PHASE 2: THẢM HỌA NHIỄU KÊNH 0 (Lò vi sóng / AP lạ đè sóng)
            data.append({
                'step': step,
                'noise_ch0': np.random.choice(high_noise_traces), # Bơm đỉnh nhiễu Kaggle vào đây
                'noise_ch1': np.random.choice(low_noise_traces),
                'noise_ch2': np.random.choice(low_noise_traces),
                'sta_count': int(np.random.choice(normal_sta_traces))
            })
        else:
            # PHASE 3: BÃO ROAMING CHUYỂN TIẾT (Hàng trăm sinh viên ập vào)
            data.append({
                'step': step,
                'noise_ch0': np.random.choice(low_noise_traces) + 0.1,
                'noise_ch1': np.random.choice(low_noise_traces) + 0.1,
                'noise_ch2': np.random.choice(low_noise_traces) + 0.1,
                'sta_count': int(np.random.choice(roaming_storm_traces)) # Bơm mật độ UJI vào đây
            })
    return pd.DataFrame(data)

# --- 2. XÂY DỰNG MÔI TRƯỜNG "CHAOS ENGINEERING" ---
class KaggleStressTestEnv(gym.Env):
    def __init__(self, kaggle_df):
        super().__init__()
        self.df = kaggle_df
        self.current_step = 0

        # Action & State khớp hoàn toàn với môi trường PPOFriendlyDigitalTwinEnv
        self.action_space = spaces.MultiDiscrete([3, 3, 7, 12, 2, 5, 2])
        self.observation_space = spaces.Box(
            low=np.array([-90.0, 0.0, 0.0, 0.0, 0, 0, 0, 0], dtype=np.float32),
            high=np.array([-30.0, 1.0, 1.0, 1.0, 300, 23, 59, 1], dtype=np.float32)
        )
        self.history = []

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = 0
        self.history = []
        return np.array([-70.0, 0.1, 0.05, 0.4, 20, 8, 15, 0], dtype=np.float32), {}

    def step(self, action):
        row = self.df.iloc[self.current_step]
        channel_idx = action[0]
        bw_idx = action[1]

        # Lấy cường độ thảm họa từ DataFrame (đã nạp Kaggle Data)
        actual_noise = [row['noise_ch0'], row['noise_ch1'], row['noise_ch2']][channel_idx]
        actual_sta = row['sta_count']

        # Logic Sinh Tồn (Vật lý)
        base_throughput = 50.0 if bw_idx == 0 else (100.0 if bw_idx == 1 else 200.0)

        # TRỪNG PHẠT AI: Nếu chọn Băng thông 80MHz khi Bão Học Sinh ập đến -> Va chạm triệt tiêu mạng
        if bw_idx == 2 and actual_sta > 100:
            actual_noise = min(0.99, actual_noise + 0.5)

        throughput = base_throughput * max(0.0, (1.0 - actual_noise))
        p_c = min(1.0, actual_noise + (actual_sta * 0.005))

        # Ghi log lịch sử cho báo cáo
        self.history.append({
            'step': self.current_step,
            'chosen_channel': channel_idx,
            'chosen_bw': [20, 40, 80][bw_idx],
            'throughput': throughput,
            'p_c': p_c,
            'env_sta': actual_sta,
            'env_noise_ch0': row['noise_ch0']
        })

        self.current_step += 1
        terminated = bool(self.current_step >= len(self.df))

        # Cập nhật State
        state = np.array([-65.0, p_c, 0.05, 0.4, actual_sta, 8, 15, 0], dtype=np.float32)
        return state, 0.0, terminated, False, {}

# --- 3. TIẾN HÀNH KIỂM THỬ MÔ HÌNH VỚI DỮ LIỆU KAGGLE ---
kaggle_data = prepare_kaggle_traces()
env = KaggleStressTestEnv(kaggle_data)

try:
    model = PPO.load("./saved_models/best_model.zip")
    print("\n[*] Đã tải thành công 'Bộ não AI' (best_model.zip)!")
except:
    print("\n[!] Lỗi: Không tìm thấy best_model.zip. Vui lòng chạy lại Bước 3.")
    exit()

obs, _ = env.reset()
done = False

print("[*] ĐANG CHẠY MÔ PHỎNG KIỂM THỬ TRACE-DRIVEN...")
while not done:
    # deterministic=True ép AI không tò mò, chỉ dùng chính sách tốt nhất để sinh tồn
    action, _states = model.predict(obs, deterministic=True)
    obs, reward, done, _, info = env.step(action)

print("✅ Hoàn tất Stress Test! Đang xuất phân tích...\n")

# --- 4. TRỰC QUAN HÓA KẾT QUẢ CHO BÁO CÁO KHKT ---
df_res = pd.DataFrame(env.history)

plt.style.use('seaborn-v0_8-darkgrid')
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9))

# BIỂU ĐỒ 1: Phản ứng của AI với Nhiễu (Đổi Kênh)
ax1.plot(df_res['step'], df_res['chosen_channel'], color='crimson', drawstyle='steps-post', linewidth=2, label='Kênh AI Chọn (Index 0,1,2)')
ax1.plot(df_res['step'], df_res['env_noise_ch0'] * 3, color='grey', alpha=0.5, linestyle='--', label='Cường độ Nhiễu Kênh 0 (Scaled)')
ax1.set_title("Hình 5A: Đánh giá khả năng Tránh Nhiễu Đồng Kênh (CCI) bằng dữ liệu RF Telemetry", fontsize=14, fontweight='bold')
ax1.set_ylabel("Chỉ số Kênh (Channel Index)")
ax1.set_yticks([0, 1, 2])
ax1.set_yticklabels(['Ch 36 (0)', 'Ch 40 (1)', 'Ch 44 (2)'])
ax1.axvspan(100, 200, color='crimson', alpha=0.1, label='Phase 2: Thảm họa Nhiễu Kênh 0')
ax1.legend(loc="upper left")

# BIỂU ĐỒ 2: Phản ứng của AI với Bão Truy cập (Co giãn Băng thông)
ax2.plot(df_res['step'], df_res['chosen_bw'], color='teal', drawstyle='steps-post', linewidth=2, label='Băng thông AI Chọn (MHz)')
ax2_twin = ax2.twinx()
ax2_twin.plot(df_res['step'], df_res['env_sta'], color='darkorange', alpha=0.6, label='Mật độ Học sinh (UJI Data)')
ax2.set_title("Hình 5B: Đánh giá khả năng Co giãn Băng thông khi xảy ra Bão Roaming (UJIIndoorLoc)", fontsize=14, fontweight='bold')
ax2.set_xlabel("Thời gian (Timesteps)")
ax2.set_ylabel("Độ rộng Kênh (MHz)", color='teal')
ax2_twin.set_ylabel("Số lượng Thiết bị", color='darkorange')
ax2.set_yticks([20, 40, 80])
ax2.axvspan(200, 300, color='darkorange', alpha=0.1, label='Phase 3: Bão Roaming Chuyển tiết')

# Gộp chú thích
lines, labels = ax2.get_legend_handles_labels()
lines2, labels2 = ax2_twin.get_legend_handles_labels()
ax2.legend(lines + lines2, labels + labels2, loc="upper left")

plt.tight_layout()
plt.savefig("./wifi_rl_logs/kaggle_chaos_test.png", dpi=300)
plt.show()  

In [ ]:
import numpy as np
import pandas as pd
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
import matplotlib.pyplot as plt

print("=== BƯỚC 4.6 (NÂNG CẤP): KIỂM THỬ ĐỘ ĐÀN HỒI (PULSE CHAOS TESTING) ===")

# --- 1. TẠO KỊCH BẢN THẢM HỌA NHIỀU ĐỢT (600 STEPS) ---
def load_multi_wave_chaos():
    data = []
    for step in range(600):
        # Mặc định mạng bình yên
        noise = [0.1, 0.1, 0.1]
        sta = 20

        # Đợt 1: Bão nhiễu tàn phá Kênh 1 (Step 100-200)
        if 100 <= step < 200:
            noise[1] = 0.95

        # Đợt 2: Bão Roaming khổng lồ (Step 300-400)
        elif 300 <= step < 400:
            sta = 160

        # Đợt 3: Bão nhiễu kép tấn công cả Kênh 0 và Kênh 1 (Step 500-600)
        elif 500 <= step < 600:
            noise[0] = 0.85
            noise[1] = 0.90

        data.append({
            'step': step,
            'noise_ch0': noise[0], 'noise_ch1': noise[1], 'noise_ch2': noise[2],
            'sta_count': sta
        })
    return pd.DataFrame(data)

# --- 2. XÂY DỰNG MÔI TRƯỜNG CHẤM ĐIỂM ---
class PulseChaosEnv(gym.Env):
    def __init__(self, df):
        super().__init__()
        self.df = df
        self.current_step = 0
        self.action_space = spaces.MultiDiscrete([3, 3, 7, 12, 2, 5, 2])
        self.observation_space = spaces.Box(
            low=np.array([-90.0, 0.0, 0.0, 0.0, 0, 0, 0, 0], dtype=np.float32),
            high=np.array([-30.0, 1.0, 1.0, 1.0, 300, 23, 59, 1], dtype=np.float32)
        )
        self.history = []

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = 0
        self.history = []
        return np.array([-70.0, 0.1, 0.05, 0.4, 20, 8, 15, 0], dtype=np.float32), {}

    def step(self, action):
        row = self.df.iloc[self.current_step]
        channel_idx = action[0]
        bw_idx = action[1]

        actual_noise = [row['noise_ch0'], row['noise_ch1'], row['noise_ch2']][channel_idx]
        actual_sta = row['sta_count']

        base_throughput = 50.0 if bw_idx == 0 else (100.0 if bw_idx == 1 else 200.0)

        if bw_idx == 2 and actual_sta > 100:
            actual_noise = min(0.99, actual_noise + 0.5)
        elif bw_idx == 1 and actual_sta > 100:
            actual_noise = min(0.99, actual_noise + 0.2)

        p_c = min(1.0, actual_noise + (actual_sta * 0.005))

        self.history.append({
            'step': self.current_step,
            'chosen_channel': channel_idx,
            'chosen_bw': [20, 40, 80][bw_idx],
            'env_sta': actual_sta,
            'env_noise_ch0': row['noise_ch0'],
            'env_noise_ch1': row['noise_ch1']
        })

        self.current_step += 1
        terminated = bool(self.current_step >= len(self.df))
        state = np.array([-65.0, p_c, 0.05, 0.4, actual_sta, 8, 15, 0], dtype=np.float32)
        return state, 0.0, terminated, False, {}

# --- 3. ĐÁNH GIÁ (INFERENCE) VỚI BEST_MODEL ---
test_df = load_multi_wave_chaos()
env = PulseChaosEnv(test_df)

try:
    # Dùng chính best_model.zip xịn nhất bạn vừa train
    model = PPO.load("./saved_models/best_model.zip")
    print("[*] Đã tải thành công 'Bộ não AI' (best_model.zip) để làm bài test Cuối kỳ!")
except:
    print("[!] Không tìm thấy best_model.zip.")
    exit()

obs, _ = env.reset()
done = False

while not done:
    # Cờ deterministic=True: Dùng phản xạ 100% thay vì tò mò random
    action, _states = model.predict(obs, deterministic=True)
    obs, reward, done, _, info = env.step(action)

# --- 4. TRỰC QUAN HÓA BẢN BÁO CÁO KHKT XUẤT SẮC ---
df_res = pd.DataFrame(env.history)
plt.style.use('seaborn-v0_8-darkgrid')
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# Biểu đồ 1: Sự đàn hồi khi Chọn Kênh
ax1.plot(df_res['step'], df_res['chosen_channel'], color='crimson', drawstyle='steps-post', linewidth=2.5, label='Kênh AI Chọn (0,1,2)')
ax1.plot(df_res['step'], df_res['env_noise_ch1'] * 2.5, color='grey', alpha=0.5, linestyle='--', label='Cường độ Nhiễu Kênh 40')
ax1.plot(df_res['step'], df_res['env_noise_ch0'] * 2.5, color='black', alpha=0.5, linestyle=':', label='Cường độ Nhiễu Kênh 36')
ax1.set_title("ĐÁNH GIÁ PHẢN XẠ CỦA AGENT RL: Khả năng Tránh Né Nhiễu Đồng Kênh (CCI) Đa Đợt", fontsize=14, fontweight='bold')
ax1.set_ylabel("Chỉ số Kênh (Index)")
ax1.set_yticks([0, 1, 2])
ax1.set_yticklabels(['Ch 36 (0)', 'Ch 40 (1)', 'Ch 44 (2)'])
ax1.axvspan(100, 200, color='red', alpha=0.1) # Bão đợt 1
ax1.axvspan(500, 600, color='red', alpha=0.1) # Bão đợt 3
ax1.legend(loc="upper left")

# Biểu đồ 2: Sự đàn hồi Băng thông
ax2.plot(df_res['step'], df_res['chosen_bw'], color='teal', drawstyle='steps-post', linewidth=2.5, label='Băng thông AI (MHz)')
ax2_twin = ax2.twinx()
ax2_twin.plot(df_res['step'], df_res['env_sta'], color='darkorange', alpha=0.6, label='Số Client truy cập')
ax2.set_title("ĐÁNH GIÁ PHẢN XẠ CỦA AGENT RL: Khả năng Tự Phục Hồi Băng Thông (Elastic Recovery)", fontsize=14, fontweight='bold')
ax2.set_xlabel("Thời gian mô phỏng (Timesteps)")
ax2.set_ylabel("Băng thông (MHz)", color='teal')
ax2_twin.set_ylabel("Số thiết bị (STA)", color='darkorange')
ax2.set_yticks([20, 40, 80])
ax2.axvspan(300, 400, color='orange', alpha=0.1) # Bão đợt 2

lines, labels = ax2.get_legend_handles_labels()
lines2, labels2 = ax2_twin.get_legend_handles_labels()
ax2.legend(lines + lines2, labels + labels2, loc="upper right")

plt.tight_layout()
plt.savefig("./wifi_rl_logs/pulse_chaos_test.png", dpi=300)
plt.show()